# Requested Individual CrossTabs (HST report) — one Excel file per crosstab

**Purpose.** Sir's request (Jul 2026): generate every crosstab listed in `CT list for codes HST to Aayushman.xlsx` as its **own Excel file**, plus the **custom summary tables** shared as images. Written for the Madhya Pradesh state-level report but runs for **any place** — just set `INPUT_FILE` and `ANALYSIS_LEVEL` in the config cell (tested here on Jalna).

**This notebook is fully standalone** — it does not modify `CrossTab_streamlined pipeline.ipynb`, its logic, or any of its output folders.

| Sheet code in the list | Data subset (identical filter to the combined pipeline) | Combined-pipeline file that already carries the same tables |
|---|---|---|
| BE_LC | Learners (dedup by `User Acc ID all`) — before exclusion | `learner_based_ct_before_exclusion_*` |
| BE_MCD | All adoptions — before exclusion | `crosstab_all_data_*` |
| AE_LC | Learners — after exclusion (combined-pipeline convention) | `learner_based_ct_after_exclusion_*` |
| AE_ANC, AE_ANC ADDNL | ANC adoptions + Included | `only_anc_ct_after_exclusion_*` |
| AE_PNCBW | PNCL5M + PNCG5M adoptions + Included | `BW_proxy_ct_after_primexc_*` |
| AE-PNCL5M | PNCL5M adoptions + Included | `pncl5m_ct_after_exclusion_*` |
| AE-PNCG5M | PNCG5M adoptions + Included | `pncg5m_ct_after_exclusion_*` |
| AE-PNCG5M BFCF | PNCG5M adoptions + Included (bf/cf category 2) | `BF_CF_Assess_pncg5m_ct_after_exclusion_*` |

> **Overlap note (checked programmatically):** all **97** listed crosstabs are *already* generated by the combined pipeline inside the section files above. This notebook re-exports each one as an **individual file** using the **same logic, filters and category orders** (so the numbers match the combined sheets), and adds the **6 custom image tables**, which the combined pipeline does **not** produce.

**Geo rule:** `ANALYSIS_LEVEL = "state"` → crosstabs use `District`; `ANALYSIS_LEVEL = "district"` → every `District` is replaced by `Blocks` (derived from `Phc Taluka_CR`) in tables, titles and file names.

**Outputs:** `output_requested_CTs/` (column-wise %) and `output_requested_CTs_2/` (row-wise %). Every file name is stamped with the input file's own code and version, e.g. `JL_140626` / `V(170626)`.

In [1]:
# ==============================================================================
# SECTION 0 — CONFIG & IMPORTS
# ==============================================================================
import os
import re
import warnings

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils.dataframe import dataframe_to_rows

warnings.filterwarnings('ignore')

# ---- What to run on ----------------------------------------------------------
INPUT_FILE = "C:/Users/AayushmanSingh/Desktop/full pipeline/Inputs/derived/UJ_010826_Exc_V(030826)_2.xlsx"   # any place's _Exc extract

# "state"    -> crosstabs use 'District'  (e.g. Madhya Pradesh, Meghalaya)
# "district" -> crosstabs use 'Blocks'    (e.g. Jalna)
ANALYSIS_LEVEL = "district"

# ---- 'Learners in final analysis' definition for the CUSTOM tables -----------
# "any_included_adoption" -> a learner is in the final analysis when AT LEAST ONE
#                            of their adoptions is Included (recommended reading
#                            of the image tables)
# "first_row_included"    -> the combined pipeline's learner-after-exclusion
#                            convention (dedup first, keep learners whose first
#                            row is Included) — matches AE_LC individual files
LEARNER_FINAL_ANALYSIS_MODE = "any_included_adoption"

# ---- Output folders (created next to this notebook) --------------------------
OUTPUT_DIR_REQ  = "output_requested_CTs"     # column-wise percentages
OUTPUT_DIR_REQ2 = "output_requested_CTs_2"   # row-wise percentages
os.makedirs(OUTPUT_DIR_REQ,  exist_ok=True)
os.makedirs(OUTPUT_DIR_REQ2, exist_ok=True)

print(f"Input: {INPUT_FILE}")
print(f"Analysis level: {ANALYSIS_LEVEL}")
print(f"Learner final-analysis mode (custom tables): {LEARNER_FINAL_ANALYSIS_MODE}")

Input: JL_140626_Exc_V(170626)_2.xlsx
Analysis level: district
Learner final-analysis mode (custom tables): any_included_adoption


---
## Section 1 — Load & prepare data
Same data steps as the combined pipeline: merge `MO` into `Others`, derive `Blocks` from `Phc Taluka_CR`, detect the Jalna input (JL code → the two Jalna adoption-group twin columns are used automatically), stamp output names with the input's code/version.

In [2]:
# ==============================================================================
# SECTION 1 — LOAD & PREPARE DATA  (mirrors the combined pipeline's data steps)
# ==============================================================================
print("Loading data...")
data1 = pd.read_excel(INPUT_FILE)
print(f"  Loaded: {data1.shape[0]} rows x {data1.shape[1]} columns")

# ---- ADD-ON (04-08-2026): apply the SAME pre-analysis filter as the main -----
# crosstab pipeline, so these individual/custom tables reconcile with the report
# tables instead of silently using a larger base. Stage 2 writes the reason in
# 'pre_analysis_exclusion' ('Reason for pre-analysis removal' on extracts from
# before 02-08-2026); rows tagged with a removal reason -- HST trainer team, no
# role/role group, no block, no training batch, Nursing staff role group, zero
# adoptions in MASD -- are dropped here exactly as the main pipeline drops them.
# Auto-inert on an extract that carries no reason column.
# Set APPLY_PRE_ANALYSIS_FILTER = False to restore the previous behaviour.
APPLY_PRE_ANALYSIS_FILTER = True

_REASON_COL = 'pre_analysis_exclusion'
if _REASON_COL not in data1.columns and 'Reason for pre-analysis removal' in data1.columns:
    _REASON_COL = 'Reason for pre-analysis removal'

if APPLY_PRE_ANALYSIS_FILTER and _REASON_COL in data1.columns:
    _pr = data1[_REASON_COL].fillna('').astype(str).str.strip()
    _n0 = len(data1)
    _l0 = data1['User Acc ID all'].astype(str).nunique() if 'User Acc ID all' in data1.columns else None
    print(f"  Pre-analysis removal: start {_n0} case row(s), {_l0} learner(s)")
    for _r, _n in _pr[~_pr.isin(['', 'Included'])].value_counts().items():
        _lr = (data1.loc[_pr.eq(_r), 'User Acc ID all'].astype(str).nunique()
               if 'User Acc ID all' in data1.columns else '?')
        print(f"    - {_r}: {_n} case row(s), {_lr} learner(s) affected")
    data1 = data1[_pr.isin(['', 'Included'])].copy()
    _l1 = data1['User Acc ID all'].astype(str).nunique() if 'User Acc ID all' in data1.columns else None
    print(f"  Analysis set: {len(data1)} case row(s), {_l1} learner(s)")
elif APPLY_PRE_ANALYSIS_FILTER:
    print("  [!] No pre-analysis reason column on this extract -> filter skipped "
          "(numbers will NOT match the main pipeline).")

IS_DISTRICT_LEVEL = ANALYSIS_LEVEL.strip().lower() == "district"
BLOCK_COL = "Blocks"
GEO_COL   = BLOCK_COL if IS_DISTRICT_LEVEL else "District"

# ---- Jalna detection (JL code in the file name) ------------------------------
IS_JALNA = "JL" in os.path.basename(INPUT_FILE).upper()
print(f"  Analysis level: {ANALYSIS_LEVEL} | Jalna (JL) input detected: {IS_JALNA}")

# ---- Merge Medical Officers into 'Others' (same as combined pipeline) --------
if 'Role Group' in data1.columns:
    _mo_mask = (data1['Role Group'].astype(str).str.strip().str.upper()
                .isin(['MO', 'MEDICAL OFFICER']))
    print(f"  Role Group: merging {int(_mo_mask.sum())} 'MO' row(s) into 'Others'")
    data1.loc[_mo_mask, 'Role Group'] = 'Others'

# ---- Derive learner 'Blocks' from 'Phc Taluka_CR' (same as combined pipeline)
def _normalise_block(v):
    if pd.isna(v):
        return 'z_NA'
    s = str(v).strip()
    s = re.sub(r'[\s\-_]*\d+$', '', s)      # strip trailing '-1' / '-2'
    s = re.sub(r'\s+', ' ', s).strip().title()
    return s if s else 'z_NA'

if 'Phc Taluka_CR' in data1.columns:
    data1[BLOCK_COL] = data1['Phc Taluka_CR'].apply(_normalise_block)
    print(f"  Derived '{BLOCK_COL}': {dict(data1[BLOCK_COL].value_counts())}")
else:
    data1[BLOCK_COL] = 'z_NA'

# ---- Jalna twin columns for the adoption-group variables ---------------------
JALNA_COL_MAP = {
    "Total Adoptions group": "Total Adoptions group jalna",
    "Total Adoptions group after exclusion": "Total Adoptions group after exclusion jalna",
}

def effective_var(name):
    # District -> Blocks at district level; adoption-group columns -> Jalna twins
    out = GEO_COL if name == "District" else name
    if IS_JALNA:
        out = JALNA_COL_MAP.get(out, out)
    return out

# ---- Output-name stamping from the input's own code / version ----------------
_base  = os.path.basename(str(INPUT_FILE))
_mcode = re.search(r'([A-Za-z]{2}_\d{6})', _base)
_mver  = re.search(r'V\((\d{6})\)', _base)
CODE_STAMP = _mcode.group(1).upper() if _mcode else 'XX_000000'
VER_STAMP  = f"V({_mver.group(1)})" if _mver else 'V(000000)'
print(f"  Output name stamp: {CODE_STAMP}_{VER_STAMP}")

# ---- Display-label renames at export time (same as combined pipeline) --------
DISPLAY_LABEL_RENAMES = {
    "visit category": "PNC visit category",
    "Adoption visit anthropometry data(Wt,Ht,zscore) not available":
        "Birth details available but no adoption details available",
    "Last visit Date not available":
        "Last visit anthropometry details available but date absent",
}

Loading data...


  Loaded: 590 rows x 1693 columns
  Analysis level: district | Jalna (JL) input detected: True
  Role Group: merging 1 'MO' row(s) into 'Others'
  Derived 'Blocks': {'Jalna': np.int64(197), 'Bhokardan': np.int64(77), 'Ambad': np.int64(75), 'Ghansawangi': np.int64(54), 'Jafrabad': np.int64(48), 'Badnapur': np.int64(45), 'Partur': np.int64(39), 'Mantha': np.int64(32), 'z_NA': np.int64(23)}
  Output name stamp: JL_140626_V(170626)


---
## Section 2 — Category display orders
Copied from the combined pipeline (`row_order`) for every variable used in the requested list, so the individual files keep the exact same category ordering as the combined sheets.

In [3]:
# ==============================================================================
# SECTION 2 — CATEGORY DISPLAY ORDERS (same values as the combined pipeline)
# ==============================================================================
CATEGORY_ORDER = {
    "Department": ['HFW', 'WCD', 'MSRLS'],
    "District": ['East Garo Hills', 'East Jaintia Hills', 'East Khasi Hills',
                 'Eastern West Khasi Hills', 'North Garo Hills', 'Ri Bhoi',
                 'South Garo Hills', 'South West Garo Hills', 'South West Khasi Hills',
                 'West Garo Hills', 'West Jaintia Hills', 'West Khasi Hills'],
    "Blocks": ['Ambad', 'Badnapur', 'Bhokardan', 'Ghansawangi',
               'Jafrabad', 'Jalna', 'Mantha', 'Partur'],
    "Role Group": ['AWW', 'AWSup', 'ANM', 'ASHA', 'CHO', 'Others', 'ASHASup',
                   'MLHP', 'Staff nurse'],
    "Total Adoptions group": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Total Adoptions group after exclusion": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Total Adoptions group jalna": ['01', '02', '03', 'More_than_03'],
    "Total Adoptions group after exclusion jalna": ['01', '02', '03', 'More_than_03'],
    "Type of adoption": ['ANC', 'PNCL5M', 'PNCG5M',
                         'Data to calculate child age of adoption is not available',
                         'Invalid difference: Baby Adoption date earlier than Mother adoption date',
                         'Mother adoption date is missing'],
    "birthweight category": ['01.5_or_less_kg', '01.51_to_02.5_kg', '02.51_to_02.7_kg',
                             '02.71_to_03.0_kg', '03.01_to_03.5_kg', 'More_than_03.5_kg'],
    "birthweight category 2": ['Less_than_01.5_kg', '01.5_to_02.49_kg', '02.5_to_02.69_kg',
                               '02.7_to_02.99_kg', '03.0_to_03.49_kg', '03.5_or_more_kg',
                               'Above_+6SD'],
    "birthweight group": ['Low Birth Weight', 'Normal Birth Weight', 'Overweight',
                          'Birth Weight is not available'],
    "adoption age classification": ['invalid_age', 'd000', 'd001_to_d015', 'd016_to_d030',
                                    'd031_to_d060', 'd061_to_d090', 'd091_to_d120',
                                    'd121_to_d150', 'd151_to_d180', 'd181_to_d210',
                                    'd211_to_d240', 'd241_to_d270', 'd271_to_d300',
                                    'd301_to_d330', 'd331_to_d365', 'd366_plus'],
    "adoption duration baby": ['Only birth data available', '01_to_30_days', '31_to_60_days',
                               '61_to_90_days', '91_days_or_more', 'negatives'],
    "visit category": ['01_visit', '02_visits', '03_visits', '04_to_05_visits',
                       '06_to_07_visits', '08_to_09_visits', '10_or_more_visits'],
    "bf assessment category": ['00_assessment', '01_assessment', '02_assessments',
                               '03_assessments', '04_to_05_assessments', '06_to_07_assessments',
                               '08_to_09_assessments', '10_or_more_assessments'],
    "cf assessment category": ['00_assessment', '01_assessment', '02_assessments',
                               '03_assessments', '04_to_05_assessments', '06_to_07_assessments',
                               '08_to_09_assessments', '10_or_more_assessments'],
    "bf assessment category 2": ['00_assessment', '01_assessment', '02_assessments',
                                 '03_assessments', '04_to_05_assessments', '06_or_more_assessments'],
    "cf assessment category 2": ['00_assessment', '01_assessment', '02_assessments',
                                 '03_assessments', '04_to_05_assessments', '06_or_more_assessments'],
    "Mother adoption till birth": ['PNC', '001_to_030', '031_to_060', '061_to_090',
                                   '091_to_120', '121_to_150', '151_to_180', '181_to_210',
                                   '211_to_240', '241_to_270', '271_to_300', 'DOB is missing',
                                   'Mother adoption date is missing',
                                   'DOB and Mother adoption both missing',
                                   'Mother adoption during pregnancy is More than 300 days- Suggestive of data entry error'],
    "ANC 60d 3visit": ['Yes', 'No'],
    "Included excluded": ['Included', 'Excluded'],
    "Primary Exclusion PNC 2": [
        "No Child ID - Child not born yet/child not followed up",
        "Birth date not available",
        "Adoption visit anthropometry data(Wt,Ht,zscore) not available",
        "Mother adoption date is after last visit date of the child - suggestive of data entry error",
        "Birth weight category is below -6SD or above +6SD",
        "Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match",
        "Date of birth in CR sheets and Visit Date 1 in CM sheet do not match",
        "Adoption date in CR sheet and Visit Date 2 in CM sheet do not match",
        "Last visit dates do not match in CR and CM sheets",
        "Wt Gain per day \u2265 180gm between Adoption and Last visit",
        "Only adoption visit occurred",
        "Only birth anthropometry details available",
        "mother_AD_DOB > 300",
        "Birth date, Adoption date, and Last visit date are the same",
        'User role belongs to excluded category',
        'User Role is not available', "Weight Zscore at LV/AV/BV is missing",
        "Height Zscore at LV/AV/BV is missing",
        "WFH Zscore at LV/AV/BV is missing",
        'Invalid difference: Baby Adoption date earlier than Mother adoption date',
        "Follow-up category is Unclassified ", "Last visit Date not available",
        'Included'],
}

---
## Section 3 — Crosstab generator & Excel writers
The generator replicates `generate_specific_crosstabs` / `generate_specific_crosstabs_rowwise` from the combined pipeline **exactly** (margins, category ordering, `z_NA` hidden from display but counted in totals, `count (percent%)` cells). The writers reproduce the combined sheets' Calibri-24 formatting.

In [4]:
# ==============================================================================
# SECTION 3 — CROSSTAB GENERATOR & EXCEL WRITERS
# ==============================================================================

def clean_label(label):
    # identical to the combined pipeline's export-time label cleaning
    if not isinstance(label, str):
        return label
    if " vs " in label:
        return " vs ".join(clean_label(p) for p in label.split(" vs "))
    label = re.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
    label = label.replace('_', ' ')
    label = re.sub(r'\s+', ' ', label)
    label = label.strip()
    return DISPLAY_LABEL_RENAMES.get(label, label)


def _is_zna(v):
    return ' '.join(str(v).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')


def make_crosstab(data, row_col, col_col, percent="column"):
    # Returns the combined 'count (percent%)' table, or None when there is no data.
    # percent="column" -> cell / column total (as in outputs/)
    # percent="row"    -> cell / row total    (as in output_2/)
    if row_col not in data.columns or col_col not in data.columns:
        return None

    count_table = pd.crosstab(data[row_col], data[col_col], dropna=True,
                              margins=True, margins_name="Total")
    if count_table.empty or count_table.shape[1] == 0:
        return None

    for axis_name, order_key in (("index", row_col), ("columns", col_col)):
        order = CATEGORY_ORDER.get(order_key)
        if order:
            labels = getattr(count_table, axis_name)
            ordered = ([x for x in order if x in labels]
                       + [x for x in labels if x not in order])
            count_table = (count_table.loc[ordered] if axis_name == "index"
                           else count_table[ordered])

    grand_total = count_table.loc["Total", "Total"]
    if percent == "column":
        percent_table = (count_table.iloc[:-1, :-1]
                         .div(count_table.loc["Total"].iloc[:-1], axis=1)
                         .replace([np.inf, -np.inf], 0).fillna(0) * 100).round(2)
        percent_table["Total"] = ((count_table["Total"] / grand_total * 100).round(2)
                                  if grand_total != 0 else 0.0)
        percent_table.loc["Total"] = 100.0
    else:
        percent_table = (count_table.iloc[:-1, :-1]
                         .div(count_table["Total"].iloc[:-1], axis=0)
                         .replace([np.inf, -np.inf], 0).fillna(0) * 100).round(2)
        percent_table["Total"] = 100.0
        percent_table.loc["Total"] = ((count_table.loc["Total"] / grand_total * 100).round(2)
                                      if grand_total != 0 else 0.0)

    zna_rows = [x for x in count_table.index if _is_zna(x)]
    zna_cols = [c for c in count_table.columns if _is_zna(c)]
    if zna_rows or zna_cols:
        count_table = count_table.drop(index=zna_rows, columns=zna_cols)
        percent_table = percent_table.drop(index=zna_rows, columns=zna_cols, errors='ignore')

    final_table = (count_table.astype(int).astype(str)
                   + " (" + percent_table.astype(str) + "%)")
    final_table.index.name = row_col
    return final_table


def _estimate_width(text, font_size=24):
    if not text:
        return 0
    return len(str(text)) * 0.9 * (font_size / 11) + 2


def _autofit(ws):
    widths = {}
    for row in ws.iter_rows():
        for cell in row:
            if cell.value:
                widths[cell.column_letter] = max(widths.get(cell.column_letter, 0),
                                                 _estimate_width(cell.value))
    for letter, w in widths.items():
        ws.column_dimensions[letter].width = w


def export_single_crosstab(table, title, file_name, out_dir, sheet_title="CrossTab"):
    # One crosstab per file, formatted exactly like the combined sheets.
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_title[:31]

    tcell = ws.cell(row=1, column=1, value=clean_label(title))
    tcell.font = Font(name='Calibri', size=24, bold=True)
    tcell.alignment = Alignment(horizontal='center')

    if table is None:
        ws.cell(row=3, column=1,
                value="No data available for this crosstab in the current input"
                ).font = Font(name='Calibri', size=24, italic=True)
    else:
        df = table.copy()
        df.columns = [clean_label(c) for c in df.columns]
        df.index = [clean_label(i) for i in df.index]
        if df.index.name:
            df.index.name = clean_label(df.index.name)
        rows = list(dataframe_to_rows(df, index=True, header=True))
        for r_idx, row in enumerate(rows):
            for c_idx, value in enumerate(row):
                cell = ws.cell(row=2 + r_idx, column=c_idx + 1, value=value)
                is_header = r_idx == 0
                is_last = r_idx == len(rows) - 1
                cell.font = Font(name='Calibri', size=24, bold=is_header or is_last)
                cell.alignment = Alignment(
                    horizontal='center' if (r_idx == 0 or c_idx == 0) else 'right')
    _autofit(ws)
    path = os.path.join(out_dir, file_name)
    wb.save(path)
    return path


def export_custom_tables(tables, path, sheet_title="Custom tables"):
    # tables: list of (title, DataFrame). A 2-level MultiIndex renders as two
    # leading columns with the block label written once per block (as in the images).
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_title[:31]
    r = 1
    for title, df in tables:
        tcell = ws.cell(row=r, column=1, value=clean_label(title))
        tcell.font = Font(name='Calibri', size=24, bold=True)
        tcell.alignment = Alignment(horizontal='left')
        r += 1

        two_level = isinstance(df.index, pd.MultiIndex)
        lead = 2 if two_level else 1
        for j, col in enumerate(df.columns):
            cell = ws.cell(row=r, column=lead + 1 + j, value=clean_label(col))
            cell.font = Font(name='Calibri', size=24, bold=True)
            cell.alignment = Alignment(horizontal='center')
        r += 1

        prev_block = None
        for i, (idx, row) in enumerate(df.iterrows()):
            if two_level:
                block, metric = idx
                if block != prev_block:
                    bcell = ws.cell(row=r, column=1, value=clean_label(block))
                    bcell.font = Font(name='Calibri', size=24, bold=True)
                    bcell.alignment = Alignment(horizontal='center')
                    prev_block = block
                mcell = ws.cell(row=r, column=2, value=clean_label(metric))
            else:
                metric = idx
                mcell = ws.cell(row=r, column=1, value=clean_label(metric))
            is_pct_row = isinstance(metric, str) and metric.strip().startswith('%')
            mcell.font = Font(name='Calibri', size=24, bold=is_pct_row)
            for j, value in enumerate(row):
                cell = ws.cell(row=r, column=lead + 1 + j, value=value)
                cell.font = Font(name='Calibri', size=24, bold=is_pct_row)
                cell.alignment = Alignment(horizontal='right')
            r += 1
        r += 2
    _autofit(ws)
    wb.save(path)
    return path

---
## Section 4 — The 97 requested crosstabs
Embedded verbatim from `CT list for codes HST to Aayushman.xlsx` (SL = row number in sir's report; each list entry becomes one Excel file in each output folder). Variable names map to data columns by replacing underscores with spaces; `District` → `Blocks` at district level; the two adoption-group variables use their Jalna twin columns on a JL input.

In [5]:
# ==============================================================================
# SECTION 4a — REQUESTED LIST + DATA SUBSETS (filters identical to the pipeline)
# ==============================================================================
REQUESTED_CTS = [
    {'sl': 1, 'sheet': 'BE_LC', 'row': 'Department', 'col': 'District', 'table': 'Department vs District'},
    {'sl': 9, 'sheet': 'BE_LC', 'row': 'Role Group', 'col': 'District', 'table': 'Role Group vs District'},
    {'sl': 21, 'sheet': 'BE_LC', 'row': 'Learner Category_2', 'col': 'District', 'table': 'Learner Category_2 vs District'},
    {'sl': 29, 'sheet': 'BE_LC', 'row': 'Learner Category_2', 'col': 'Role Group', 'table': 'Learner Category_2 vs Role Group'},
    {'sl': 95, 'sheet': 'BE_MCD', 'row': 'Department', 'col': 'District', 'table': 'Department vs District'},
    {'sl': 103, 'sheet': 'BE_MCD', 'row': 'Role Group', 'col': 'District', 'table': 'Role Group vs District'},
    {'sl': 115, 'sheet': 'BE_MCD', 'row': 'Learner Category_2', 'col': 'District', 'table': 'Learner Category_2 vs District'},
    {'sl': 151, 'sheet': 'BE_MCD', 'row': 'Primary Exclusion_PNC_2', 'col': 'District', 'table': 'Primary Exclusion_PNC_2 vs District'},
    {'sl': 169, 'sheet': 'BE_MCD', 'row': 'Primary Exclusion_PNC_2', 'col': 'Department', 'table': 'Primary Exclusion_PNC_2 vs Department'},
    {'sl': 205, 'sheet': 'BE_MCD', 'row': 'Primary Exclusion_PNC_2', 'col': 'Learner Category_2', 'table': 'Primary Exclusion_PNC_2 vs Learner Category_2'},
    {'sl': 1180, 'sheet': 'AE_LC', 'row': 'Total_Adoptions_group_after_exclusion', 'col': 'District', 'table': 'Total_Adoptions_group_after_exclusion vs District'},
    {'sl': 1188, 'sheet': 'AE_LC', 'row': 'Total_Adoptions_group_after_exclusion', 'col': 'Learner Category_2', 'table': 'Total_Adoptions_group_after_exclusion vs Learner Category_2'},
    {'sl': 1232, 'sheet': 'AE_ANC', 'row': 'birthweight_category_2', 'col': 'ANC_protein_category', 'table': 'birthweight_category_2 vs ANC_protein_category'},
    {'sl': 1244, 'sheet': 'AE_ANC', 'row': 'birthweight_category_2', 'col': 'PNC_protein_category', 'table': 'birthweight_category_2 vs PNC_protein_category'},
    {'sl': 1256, 'sheet': 'AE_ANC', 'row': 'birthweight_category_2', 'col': 'Mother adoption till birth', 'table': 'birthweight_category_2 vs Mother adoption till birth'},
    {'sl': 1268, 'sheet': 'AE_ANC', 'row': 'birthweight_group', 'col': 'ANC_protein_category', 'table': 'birthweight_group vs ANC_protein_category'},
    {'sl': 1286, 'sheet': 'AE_ANC', 'row': 'birthweight_group', 'col': 'Mother adoption till birth', 'table': 'birthweight_group vs Mother adoption till birth'},
    {'sl': 1295, 'sheet': 'AE_ANC', 'row': 'adoption_age_classification', 'col': 'District', 'table': 'adoption_age_classification vs District'},
    {'sl': 1312, 'sheet': 'AE_ANC', 'row': 'adoption_age_classification', 'col': 'Role Group', 'table': 'adoption_age_classification vs Role Group'},
    {'sl': 1329, 'sheet': 'AE_ANC', 'row': 'adoption_age_classification', 'col': 'Learner Category_2', 'table': 'adoption_age_classification vs Learner Category_2'},
    {'sl': 1409, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'District', 'table': 'visit_category vs District'},
    {'sl': 1420, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'Role Group', 'table': 'visit_category vs Role Group'},
    {'sl': 1431, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'Learner Category_2', 'table': 'visit_category vs Learner Category_2'},
    {'sl': 1464, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'adoption_age_classification', 'table': 'visit_category vs adoption_age_classification'},
    {'sl': 1475, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'adoption_duration_baby', 'table': 'visit_category vs adoption_duration_baby'},
    {'sl': 1508, 'sheet': 'AE_ANC', 'row': 'visit_category', 'col': 'birthweight_group', 'table': 'visit_category vs birthweight_group'},
    {'sl': 1519, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'District', 'table': 'bf_assessment_category vs District'},
    {'sl': 1533, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'Role Group', 'table': 'bf_assessment_category vs Role Group'},
    {'sl': 1547, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'Learner Category_2', 'table': 'bf_assessment_category vs Learner Category_2'},
    {'sl': 1575, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'adoption_age_classification', 'table': 'bf_assessment_category vs adoption_age_classification'},
    {'sl': 1589, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'adoption_duration_baby', 'table': 'bf_assessment_category vs adoption_duration_baby'},
    {'sl': 1617, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'birthweight_category_2', 'table': 'bf_assessment_category vs birthweight_category_2'},
    {'sl': 1631, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'birthweight_group', 'table': 'bf_assessment_category vs birthweight_group'},
    {'sl': 1645, 'sheet': 'AE_ANC', 'row': 'bf_assessment_category', 'col': 'visit_category', 'table': 'bf_assessment_category vs visit_category'},
    {'sl': 2208, 'sheet': 'AE_PNCBW', 'row': 'birthweight_category_2', 'col': 'District', 'table': 'birthweight_category_2 vs District'},
    {'sl': 2220, 'sheet': 'AE_PNCBW', 'row': 'birthweight_group', 'col': 'District', 'table': 'birthweight_group vs District'},
    {'sl': 2274, 'sheet': 'AE_PNCBW', 'row': 'birthweight_category_2', 'col': 'Learner Category_2', 'table': 'birthweight_category_2 vs Learner Category_2'},
    {'sl': 2286, 'sheet': 'AE_PNCBW', 'row': 'birthweight_group', 'col': 'Learner Category_2', 'table': 'birthweight_group vs Learner Category_2'},
    {'sl': 2307, 'sheet': 'AE_PNCBW', 'row': 'birthweight_category_2', 'col': 'Type_of_adoption', 'table': 'birthweight_category_2 vs Type_of_adoption'},
    {'sl': 2319, 'sheet': 'AE_PNCBW', 'row': 'birthweight_group', 'col': 'Type_of_adoption', 'table': 'birthweight_group vs Type_of_adoption'},
    {'sl': 2536, 'sheet': 'AE_PNCBW', 'row': 'birthweight_category_2', 'col': 'ANC_protein_category', 'table': 'birthweight_category_2 vs ANC_protein_category'},
    {'sl': 2548, 'sheet': 'AE_PNCBW', 'row': 'birthweight_group', 'col': 'ANC_protein_category', 'table': 'birthweight_group vs ANC_protein_category'},
    {'sl': 2604, 'sheet': 'AE_ANC ADDNL', 'row': 'birthweight_category_2', 'col': 'Learner Category_2', 'table': 'birthweight_category_2 vs Learner Category_2'},
    {'sl': 2616, 'sheet': 'AE_ANC ADDNL', 'row': 'birthweight_category_2', 'col': 'ANC_60d_3visit', 'table': 'birthweight_category_2 vs ANC_60d_3visit'},
    {'sl': 2626, 'sheet': 'AE_ANC ADDNL', 'row': 'birthweight_group', 'col': 'Learner Category_2', 'table': 'birthweight_group vs Learner Category_2'},
    {'sl': 2635, 'sheet': 'AE_ANC ADDNL', 'row': 'birthweight_group', 'col': 'ANC_60d_3visit', 'table': 'birthweight_group vs ANC_60d_3visit'},
    {'sl': 2642, 'sheet': 'AE_ANC ADDNL', 'row': 'ANC_60d_3visit', 'col': 'Learner Category_2', 'table': 'ANC_60d_3visit vs Learner Category_2'},
    {'sl': 2648, 'sheet': 'AE-PNCL5M', 'row': 'adoption_age_classification', 'col': 'District', 'table': 'adoption_age_classification vs District'},
    {'sl': 2660, 'sheet': 'AE-PNCL5M', 'row': 'adoption_age_classification', 'col': 'Role Group', 'table': 'adoption_age_classification vs Role Group'},
    {'sl': 2672, 'sheet': 'AE-PNCL5M', 'row': 'adoption_age_classification', 'col': 'Learner Category_2', 'table': 'adoption_age_classification vs Learner Category_2'},
    {'sl': 2742, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'District', 'table': 'visit_category vs District'},
    {'sl': 2753, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'Role Group', 'table': 'visit_category vs Role Group'},
    {'sl': 2764, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'Learner Category_2', 'table': 'visit_category vs Learner Category_2'},
    {'sl': 2797, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'adoption_age_classification', 'table': 'visit_category vs adoption_age_classification'},
    {'sl': 2808, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'adoption_duration_baby', 'table': 'visit_category vs adoption_duration_baby'},
    {'sl': 2830, 'sheet': 'AE-PNCL5M', 'row': 'visit_category', 'col': 'birthweight_category_2', 'table': 'visit_category vs birthweight_category_2'},
    {'sl': 2841, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'District', 'table': 'bf_assessment_category vs District'},
    {'sl': 2855, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'Role Group', 'table': 'bf_assessment_category vs Role Group'},
    {'sl': 2869, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'Learner Category_2', 'table': 'bf_assessment_category vs Learner Category_2'},
    {'sl': 2953, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'adoption_age_classification', 'table': 'bf_assessment_category vs adoption_age_classification'},
    {'sl': 2981, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'adoption_duration_baby', 'table': 'bf_assessment_category vs adoption_duration_baby'},
    {'sl': 3023, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'birthweight_category_2', 'table': 'bf_assessment_category vs birthweight_category_2'},
    {'sl': 3037, 'sheet': 'AE-PNCL5M', 'row': 'bf_assessment_category', 'col': 'visit_category', 'table': 'bf_assessment_category vs visit_category'},
    {'sl': 3501, 'sheet': 'AE-PNCG5M', 'row': 'adoption_age_classification', 'col': 'District', 'table': 'adoption_age_classification vs District'},
    {'sl': 3515, 'sheet': 'AE-PNCG5M', 'row': 'adoption_age_classification', 'col': 'Role Group', 'table': 'adoption_age_classification vs Role Group'},
    {'sl': 3529, 'sheet': 'AE-PNCG5M', 'row': 'adoption_age_classification', 'col': 'Learner Category_2', 'table': 'adoption_age_classification vs Learner Category_2'},
    {'sl': 3595, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'District', 'table': 'visit_category vs District'},
    {'sl': 3606, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'Role Group', 'table': 'visit_category vs Role Group'},
    {'sl': 3617, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'Learner Category_2', 'table': 'visit_category vs Learner Category_2'},
    {'sl': 3648, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'adoption_age_classification', 'table': 'visit_category vs adoption_age_classification'},
    {'sl': 3659, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'adoption_duration_baby', 'table': 'visit_category vs adoption_duration_baby'},
    {'sl': 3681, 'sheet': 'AE-PNCG5M', 'row': 'visit_category', 'col': 'birthweight_category_2', 'table': 'visit_category vs birthweight_category_2'},
    {'sl': 3692, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'District', 'table': 'bf_assessment_category vs District'},
    {'sl': 3706, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'Role Group', 'table': 'bf_assessment_category vs Role Group'},
    {'sl': 3720, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'Learner Category_2', 'table': 'bf_assessment_category vs Learner Category_2'},
    {'sl': 3734, 'sheet': 'AE-PNCG5M', 'row': 'cf_assessment_category', 'col': 'District', 'table': 'cf_assessment_category vs District'},
    {'sl': 3748, 'sheet': 'AE-PNCG5M', 'row': 'cf_assessment_category', 'col': 'Role Group', 'table': 'cf_assessment_category vs Role Group'},
    {'sl': 3762, 'sheet': 'AE-PNCG5M', 'row': 'cf_assessment_category', 'col': 'Learner Category_2', 'table': 'cf_assessment_category vs Learner Category_2'},
    {'sl': 3804, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'adoption_age_classification', 'table': 'bf_assessment_category vs adoption_age_classification'},
    {'sl': 3818, 'sheet': 'AE-PNCG5M', 'row': 'cf_assessment_category', 'col': 'adoption_age_classification', 'table': 'cf_assessment_category vs adoption_age_classification'},
    {'sl': 3832, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'adoption_duration_baby', 'table': 'bf_assessment_category vs adoption_duration_baby'},
    {'sl': 3846, 'sheet': 'AE-PNCG5M', 'row': 'cf_assessment_category', 'col': 'adoption_duration_baby', 'table': 'cf_assessment_category vs adoption_duration_baby'},
    {'sl': 3888, 'sheet': 'AE-PNCG5M', 'row': 'bf_assessment_category', 'col': 'visit_category', 'table': 'bf_assessment_category vs visit_category'},
    {'sl': 4325, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'District', 'table': 'bf_assessment_category_2 vs District'},
    {'sl': 4337, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'Role Group', 'table': 'bf_assessment_category_2 vs Role Group'},
    {'sl': 4349, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'Learner Category_2', 'table': 'bf_assessment_category_2 vs Learner Category_2'},
    {'sl': 4361, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'District', 'table': 'cf_assessment_category_2 vs District'},
    {'sl': 4373, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'Role Group', 'table': 'cf_assessment_category_2 vs Role Group'},
    {'sl': 4385, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'Learner Category_2', 'table': 'cf_assessment_category_2 vs Learner Category_2'},
    {'sl': 4421, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'adoption_age_classification', 'table': 'bf_assessment_category_2 vs adoption_age_classification'},
    {'sl': 4433, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'adoption_age_classification', 'table': 'cf_assessment_category_2 vs adoption_age_classification'},
    {'sl': 4445, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'adoption_duration_baby', 'table': 'bf_assessment_category_2 vs adoption_duration_baby'},
    {'sl': 4457, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'adoption_duration_baby', 'table': 'cf_assessment_category_2 vs adoption_duration_baby'},
    {'sl': 4469, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'birthweight_category', 'table': 'bf_assessment_category_2 vs birthweight_category'},
    {'sl': 4481, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'birthweight_category_2', 'table': 'bf_assessment_category_2 vs birthweight_category_2'},
    {'sl': 4493, 'sheet': 'AE-PNCG5M BFCF', 'row': 'bf_assessment_category_2', 'col': 'visit_category', 'table': 'bf_assessment_category_2 vs visit_category'},
    {'sl': 4505, 'sheet': 'AE-PNCG5M BFCF', 'row': 'cf_assessment_category_2', 'col': 'visit_category', 'table': 'cf_assessment_category_2 vs visit_category'},
]

print(f"Requested crosstabs: {len(REQUESTED_CTS)}")

# ---- Data subsets (exactly the combined pipeline's filters) ------------------
df_learners = data1.drop_duplicates(subset=['User Acc ID all'])
df_learners = df_learners[df_learners['User Acc ID all'].notna()]
df_learners_incl = df_learners[df_learners['Primary Exclusion PNC 2'] == 'Included']

df_anc_incl = data1[(data1['Type of adoption'] == 'ANC') &
                    (data1['Exclusion Reason PNC 2'] == 'Included')]
df_pncl5m_incl = data1[(data1['Type of adoption'] == 'PNCL5M') &
                       (data1['Exclusion Reason PNC 2'] == 'Included')]
df_pncg5m_incl = data1[(data1['Type of adoption'] == 'PNCG5M') &
                       (data1['Exclusion Reason PNC 2'] == 'Included')]
df_pnc_both_incl = data1[((data1['Type of adoption'] == 'PNCL5M') |
                          (data1['Type of adoption'] == 'PNCG5M')) &
                         (data1['Primary Exclusion PNC 2'] == 'Included')]

SHEET_DATA = {
    'BE_LC':          (df_learners,      'Learners (dedup) — before exclusion'),
    'BE_MCD':         (data1,            'All adoptions — before exclusion'),
    'AE_LC':          (df_learners_incl, 'Learners (dedup) — after exclusion'),
    'AE_ANC':         (df_anc_incl,      'ANC + Included'),
    'AE_ANC ADDNL':   (df_anc_incl,      'ANC + Included'),
    'AE_PNCBW':       (df_pnc_both_incl, 'PNCL5M+PNCG5M + Included'),
    'AE-PNCL5M':      (df_pncl5m_incl,   'PNCL5M + Included'),
    'AE-PNCG5M':      (df_pncg5m_incl,   'PNCG5M + Included'),
    'AE-PNCG5M BFCF': (df_pncg5m_incl,   'PNCG5M + Included'),
}

for _code, (_df, _desc) in SHEET_DATA.items():
    print(f"  {_code:<15s} {_desc:<40s} n={len(_df)}")

# ---- Validation: every requested variable must exist in its subset -----------
_missing = []
for _req in REQUESTED_CTS:
    _df = SHEET_DATA[_req['sheet']][0]
    for _v in (_req['row'], _req['col']):
        _eff = effective_var(_v.replace('_', ' '))
        if _eff not in _df.columns:
            _missing.append((_req['sl'], _req['sheet'], _v, _eff))
if _missing:
    print("\nWARNING — variables not found in the data:")
    for _m in _missing:
        print("  ", _m)
else:
    print("\nAll requested variables found in the data.")

Requested crosstabs: 97


  BE_LC           Learners (dedup) — before exclusion      n=192
  BE_MCD          All adoptions — before exclusion         n=590
  AE_LC           Learners (dedup) — after exclusion       n=50
  AE_ANC          ANC + Included                           n=2
  AE_ANC ADDNL    ANC + Included                           n=2
  AE_PNCBW        PNCL5M+PNCG5M + Included                 n=214
  AE-PNCL5M       PNCL5M + Included                        n=125
  AE-PNCG5M       PNCG5M + Included                        n=89
  AE-PNCG5M BFCF  PNCG5M + Included                        n=89

All requested variables found in the data.


In [6]:
# ==============================================================================
# SECTION 4b — GENERATE ONE EXCEL FILE PER REQUESTED CROSSTAB (both % versions)
# ==============================================================================
def _sanitize(part):
    return re.sub(r'[^A-Za-z0-9()._-]+', '_', str(part).strip()).strip('_')

generated, empty_tables = [], []
for seq, req in enumerate(REQUESTED_CTS, start=1):
    subset, _ = SHEET_DATA[req['sheet']]
    eff_row = effective_var(req['row'].replace('_', ' '))
    eff_col = effective_var(req['col'].replace('_', ' '))
    title = f"{eff_row} vs {eff_col}"
    fname = (f"{seq:02d}_{_sanitize(req['sheet'])}_{_sanitize(eff_row)}"
             f"_vs_{_sanitize(eff_col)}_{CODE_STAMP}_{VER_STAMP}.xlsx")

    for pct_dir, out_dir in (("column", OUTPUT_DIR_REQ), ("row", OUTPUT_DIR_REQ2)):
        table = make_crosstab(subset, eff_row, eff_col, percent=pct_dir)
        export_single_crosstab(table, title, fname, out_dir,
                               sheet_title=req['sheet'])
        if table is None and pct_dir == "column":
            empty_tables.append((req['sl'], req['sheet'], title))
    generated.append(fname)
    print(f"  [{seq:02d}/{len(REQUESTED_CTS)}] {req['sheet']:<15s} {title}")

print(f"\nDone: {len(generated)} files in '{OUTPUT_DIR_REQ}' (column-wise %)"
      f" and {len(generated)} in '{OUTPUT_DIR_REQ2}' (row-wise %).")
if empty_tables:
    print("\nTables with NO data in this input (file written with a note):")
    for _t in empty_tables:
        print("  ", _t)

  [01/97] BE_LC           Department vs Blocks


  [02/97] BE_LC           Role Group vs Blocks


  [03/97] BE_LC           Learner Category 2 vs Blocks
  [04/97] BE_LC           Learner Category 2 vs Role Group


  [05/97] BE_MCD          Department vs Blocks
  [06/97] BE_MCD          Role Group vs Blocks


  [07/97] BE_MCD          Learner Category 2 vs Blocks
  [08/97] BE_MCD          Primary Exclusion PNC 2 vs Blocks


...

  [85/97] AE-PNCG5M BFCF  bf assessment category 2 vs Role Group
  [86/97] AE-PNCG5M BFCF  bf assessment category 2 vs Learner Category 2
  [87/97] AE-PNCG5M BFCF  cf assessment category 2 vs Blocks


  [88/97] AE-PNCG5M BFCF  cf assessment category 2 vs Role Group
  [89/97] AE-PNCG5M BFCF  cf assessment category 2 vs Learner Category 2


  [90/97] AE-PNCG5M BFCF  bf assessment category 2 vs adoption age classification
  [91/97] AE-PNCG5M BFCF  cf assessment category 2 vs adoption age classification
  [92/97] AE-PNCG5M BFCF  bf assessment category 2 vs adoption duration baby


  [93/97] AE-PNCG5M BFCF  cf assessment category 2 vs adoption duration baby
  [94/97] AE-PNCG5M BFCF  bf assessment category 2 vs birthweight category
  [95/97] AE-PNCG5M BFCF  bf assessment category 2 vs birthweight category 2


  [96/97] AE-PNCG5M BFCF  bf assessment category 2 vs visit category
  [97/97] AE-PNCG5M BFCF  cf assessment category 2 vs visit category

Done: 97 files in 'output_requested_CTs' (column-wise %) and 97 in 'output_requested_CTs_2' (row-wise %).


---
## Section 5 — Custom report tables (from the shared images)
Six tables the combined pipeline does **not** produce. Structure follows the images; `District` becomes `Blocks` on a district-level run; on a JL input the Jalna adoption-group twin columns (01 / 02 / 03 / More_than_03) are used automatically.

1. **Adoption retention by cadre** — total adoptions vs adoptions in final analysis, by Role Group
2. **Learner retention by geography & department** — F2F learners vs learners in final analysis (HFW / WCD / Total)
3. **Learner retention by cadre** — same, by Role Group
4. **Distribution of Total Adoptions group before vs after exclusion** (learner level, column-wise %)
5. **Distribution of Total Adoptions group before vs after exclusion — MT + FL vs Other learners**
6. **Adoption retention by geography & department** — total adoptions vs adoptions in final analysis (HFW / WCD / Total)

`z_NA` categories are hidden from display but stay inside every Total (combined-pipeline convention) — this is why department rows can sum to less than the Total row, exactly as in the images.

In [7]:
# ==============================================================================
# SECTION 5 — CUSTOM REPORT TABLES
# ==============================================================================
_incl_mask = data1['Primary Exclusion PNC 2'] == 'Included'
adoptions_all, adoptions_incl = data1, data1[_incl_mask]

# 'Learners in final analysis' per the configured definition
if LEARNER_FINAL_ANALYSIS_MODE == "any_included_adoption":
    _final_ids = set(adoptions_incl['User Acc ID all'].dropna())
    learners_final = df_learners[df_learners['User Acc ID all'].isin(_final_ids)]
else:
    learners_final = df_learners_incl

print(f"F2F learners (in extract): {len(df_learners)}")
print(f"Learners in final analysis [{LEARNER_FINAL_ANALYSIS_MODE}]: {len(learners_final)}")
print(f"  (alternative definition would give: "
      f"{len(df_learners_incl) if LEARNER_FINAL_ANALYSIS_MODE == 'any_included_adoption' else len(set(adoptions_incl['User Acc ID all'].dropna()))})")
print(f"Total adoptions: {len(adoptions_all)} | Adoptions in final analysis: {len(adoptions_incl)}")

TG_BEFORE = effective_var('Total Adoptions group')
TG_AFTER  = effective_var('Total Adoptions group after exclusion')

def _ordered_cats(series, var):
    # custom tables only: match categories against CATEGORY_ORDER ignoring
    # case/whitespace, so raw variants like 'ASHA ' / 'ASHA Sup' keep the
    # intended cadre order. The 97 individual CT files do not use this helper
    # and keep the combined pipeline's exact ordering behaviour.
    cats = [c for c in series.dropna().unique() if not _is_zna(c)]
    pref = CATEGORY_ORDER.get(var, [])
    def _norm(v):
        return re.sub(r'\s+', '', str(v)).lower()
    pref_norm = [_norm(p) for p in pref]
    def _pos(c):
        n = _norm(c)
        return pref_norm.index(n) if n in pref_norm else None
    matched = sorted([c for c in cats if _pos(c) is not None], key=_pos)
    unmatched = sorted([c for c in cats if _pos(c) is None], key=str)
    return matched + unmatched

def _fmt_pct(num, den):
    return f"{(num / den * 100):.2f}%" if den else "0.00%"

def retention_rows(all_df, final_df, group_col, cats, metric_labels):
    r_tot = [int((all_df[group_col] == c).sum()) for c in cats] + [len(all_df)]
    r_fin = [int((final_df[group_col] == c).sum()) for c in cats] + [len(final_df)]
    r_pct = [_fmt_pct(f, t) for f, t in zip(r_fin, r_tot)]
    return [r_tot, r_fin, r_pct]

def retention_table(all_df, final_df, group_col, metric_labels):
    cats = _ordered_cats(all_df[group_col], group_col)
    rows = retention_rows(all_df, final_df, group_col, cats, metric_labels)
    return pd.DataFrame(rows, index=metric_labels, columns=cats + ['Total'])

def dept_retention_table(all_df, final_df, geo_col, metric_labels):
    cats = _ordered_cats(all_df[geo_col], geo_col)
    depts = _ordered_cats(all_df['Department'], 'Department')
    blocks, index = [], []
    for d in depts:
        rows = retention_rows(all_df[all_df['Department'] == d],
                              final_df[final_df['Department'] == d],
                              geo_col, cats, metric_labels)
        blocks += rows
        index += [(f"{d} learners", m) for m in metric_labels]
    rows = retention_rows(all_df, final_df, geo_col, cats, metric_labels)
    blocks += rows
    index += [("Total learners", m) for m in metric_labels]
    return pd.DataFrame(blocks, index=pd.MultiIndex.from_tuples(index),
                        columns=cats + ['Total'])

ADOPTION_METRICS = ["Total adoptions", "Adoptions in final analysis",
                    "% of total adoptions included in analysis"]
LEARNER_METRICS  = ["F2F Learners", "Learners in final analysis",
                    "% of F2F learners included in analysis"]

ct_adopt_cadre = retention_table(adoptions_all, adoptions_incl, 'Role Group', ADOPTION_METRICS)
ct_adopt_geo   = dept_retention_table(adoptions_all, adoptions_incl, GEO_COL, ADOPTION_METRICS)
ct_learn_cadre = retention_table(df_learners, learners_final, 'Role Group', LEARNER_METRICS)
ct_learn_geo   = dept_retention_table(df_learners, learners_final, GEO_COL, LEARNER_METRICS)

# ---- Distribution of Total Adoptions group before vs after exclusion ---------
_after_per_learner = adoptions_incl.drop_duplicates(subset=['User Acc ID all'])

# ==============================================================================
# LEARNER-CATEGORY INTEGRITY GUARD (added Jul 2026)
# The MT+FL / Other split reads each learner's category from ONE deduplicated
# row. The 'before exclusion' column dedups the FIRST overall row per learner;
# the 'after exclusion' column dedups the FIRST *Included* row per learner.
# Those can be different rows, so a learner whose 'Learner Category 2' is not the
# same on every row can be counted in different groups before vs after -- which
# can make an 'after exclusion' subgroup LARGER than 'before exclusion', which is
# logically impossible for a subset. If that happens we DO NOT emit the (wrong)
# custom report sheet: we flag every learner that carries 2+ distinct
# 'Learner Category 2' values, write ONLY the per-learner frequency list, and
# STOP the analysis so the numbers can be fixed at source first.
# ==============================================================================
_LC_COL = 'Learner Category 2'
_lc_present = [g for g in ['MT + FL', 'Other']
               if g in set(df_learners[_LC_COL].dropna())]
_impossible = []
for _g in _lc_present:
    _before_n = int((df_learners[_LC_COL] == _g).sum())          # F2F learners (before)
    _after_n  = int((_after_per_learner[_LC_COL] == _g).sum())   # learners in final analysis (after)
    if _after_n > _before_n:
        _impossible.append((_g, _before_n, _after_n))

if _impossible:
    # learners carrying 2+ distinct Learner Category 2 values (the root cause)
    _nuniq = (data1.dropna(subset=['User Acc ID all'])
                    .groupby('User Acc ID all')[_LC_COL].nunique())
    _bad_ids = _nuniq[_nuniq >= 2].index.tolist()
    _freq = (data1[data1['User Acc ID all'].isin(_bad_ids)]
                 .groupby(['User Acc ID all', _LC_COL]).size()
                 .unstack(fill_value=0))
    _freq['total rows'] = _freq.sum(axis=1)
    _freq = _freq.reset_index()

    print("\n" + "!" * 72)
    print("ANALYSIS STOPPED -- logically impossible learner count detected")
    print("!" * 72)
    for _g, _b, _a in _impossible:
        print(f"  '{_g}' learners: BEFORE exclusion = {_b}, AFTER exclusion = {_a}"
              f"  (after > before is impossible for a subset)")
    print(f"\nCause: {len(_bad_ids)} learner(s) carry 2+ different '{_LC_COL}' "
          f"values across their rows, so they land in different groups before vs "
          f"after. Per-learner frequency of each value:")
    print(_freq.to_string(index=False))

    _flag_fname = (f"FLAGGED_inconsistent_{_LC_COL.replace(' ', '_')}"
                   f"_{CODE_STAMP}_{VER_STAMP}.xlsx")
    for _dir in (OUTPUT_DIR_REQ, OUTPUT_DIR_REQ2):
        _freq.to_excel(os.path.join(_dir, _flag_fname), index=False)
        print(f"Flagged list written: {os.path.join(_dir, _flag_fname)}")

    raise RuntimeError(
        f"Inconsistent '{_LC_COL}' for {len(_bad_ids)} learner(s): each learner "
        f"must have a single '{_LC_COL}'. Custom report NOT generated -- fix the "
        f"flagged learners in the source sheet and re-run. "
        f"(Flagged list: {_flag_fname})")

def tg_distribution(before_df, after_df):
    cats = CATEGORY_ORDER.get(TG_BEFORE, [])
    cats = ([c for c in cats if c in set(before_df[TG_BEFORE].dropna())
             | set(after_df[TG_AFTER].dropna())]
            or _ordered_cats(before_df[TG_BEFORE], TG_BEFORE))
    n_b, n_a = len(before_df), len(after_df)
    col_b = f"Before exclusion — F2F Learners (n={n_b})"
    col_a = f"After exclusion — Learners in final analysis (n={n_a})"
    return pd.DataFrame(
        {col_b: [_fmt_pct(int((before_df[TG_BEFORE] == c).sum()), n_b) for c in cats],
         col_a: [_fmt_pct(int((after_df[TG_AFTER] == c).sum()), n_a) for c in cats]},
        index=cats)

ct_tg_dist = tg_distribution(df_learners, _after_per_learner)

# MT + FL vs Other learners split (Learner Category 2)
_lc_groups = [g for g in ['MT + FL', 'Other']
              if g in set(df_learners['Learner Category 2'].dropna())]
_frames = []
for g in _lc_groups + ['Total']:
    if g == 'Total':
        b_df, a_df = df_learners, _after_per_learner
        label = 'Total'
    else:
        b_df = df_learners[df_learners['Learner Category 2'] == g]
        a_df = _after_per_learner[_after_per_learner['Learner Category 2'] == g]
        label = f"{g} learners"
    d = tg_distribution(b_df, a_df)
    d.columns = [f"{label} — {c}" for c in d.columns]
    _frames.append(d)
ct_tg_dist_lc = pd.concat(_frames, axis=1).fillna("0.00%")

GEO_TITLE = 'block' if IS_DISTRICT_LEVEL else 'district'
CUSTOM_TABLES = [
    ("Adoption retention by cadre (Role Group)", ct_adopt_cadre),
    (f"Learner retention by {GEO_TITLE} and department", ct_learn_geo),
    ("Learner retention by cadre (Role Group)", ct_learn_cadre),
    ("Distribution of Total Adoptions group before and after exclusion (learner level)",
     ct_tg_dist),
    ("Distribution of Total Adoptions group before and after exclusion — MT + FL vs Other learners",
     ct_tg_dist_lc),
    (f"Adoption retention by {GEO_TITLE} and department", ct_adopt_geo),
]

_custom_fname = f"custom_report_tables_{CODE_STAMP}_{VER_STAMP}.xlsx"
for _dir in (OUTPUT_DIR_REQ, OUTPUT_DIR_REQ2):
    _p = export_custom_tables(CUSTOM_TABLES, os.path.join(_dir, _custom_fname))
    print(f"Exported: {_p} ({len(CUSTOM_TABLES)} tables)")

for _t, _df in CUSTOM_TABLES:
    print("\n" + "-" * 70)
    print(_t)
    print(_df.to_string())

F2F learners (in extract): 192
Learners in final analysis [any_included_adoption]: 127
  (alternative definition would give: 50)
Total adoptions: 590 | Adoptions in final analysis: 216


Exported: output_requested_CTs\custom_report_tables_JL_140626_V(170626).xlsx (6 tables)
Exported: output_requested_CTs_2\custom_report_tables_JL_140626_V(170626).xlsx (6 tables)

----------------------------------------------------------------------
Adoption retention by cadre (Role Group)
                                              AWW     ANM   ASHA      CHO  Others ASHA Sup Staff nurse   Total
Total adoptions                                71     147     130     131       7       21          78     590
Adoptions in final analysis                    27      64      41      55       2       10          17     216
% of total adoptions included in analysis  38.03%  43.54%  31.54%  41.98%  28.57%   47.62%      21.79%  36.61%

----------------------------------------------------------------------
Learner retention by block and department
                                                        Ambad Badnapur Bhokardan Ghansawangi Jafrabad   Jalna  Mantha  Partur   Total
HFW learners   F2

In [8]:
# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
for _dir in (OUTPUT_DIR_REQ, OUTPUT_DIR_REQ2):
    _files = [f for f in os.listdir(_dir) if f.endswith('.xlsx')]
    print(f"{_dir}: {len(_files)} Excel files "
          f"({len(REQUESTED_CTS)} requested CTs + 1 custom-tables file)")

# consistency check: after-exclusion adoption-group value constant per learner
_chk = adoptions_incl.groupby('User Acc ID all')[TG_AFTER].nunique()
print(f"\nCheck — '{TG_AFTER}' constant per learner among included rows: "
      f"{'OK' if (_chk <= 1).all() else 'MISMATCH — inspect!'}")
print(f"Included adoptions: {len(adoptions_incl)} / {len(data1)}")
print("\nNOTE: 'Learners in final analysis' in the CUSTOM tables uses "
      f"LEARNER_FINAL_ANALYSIS_MODE = '{LEARNER_FINAL_ANALYSIS_MODE}'. The AE_LC "
      "individual files instead follow the combined pipeline's learner-after-"
      "exclusion convention so their numbers keep matching the combined sheets.")
print("\nALL DONE.")

output_requested_CTs: 98 Excel files (97 requested CTs + 1 custom-tables file)
output_requested_CTs_2: 98 Excel files (97 requested CTs + 1 custom-tables file)

Check — 'Total Adoptions group after exclusion jalna' constant per learner among included rows: OK
Included adoptions: 216 / 590

NOTE: 'Learners in final analysis' in the CUSTOM tables uses LEARNER_FINAL_ANALYSIS_MODE = 'any_included_adoption'. The AE_LC individual files instead follow the combined pipeline's learner-after-exclusion convention so their numbers keep matching the combined sheets.

ALL DONE.
